<h1>
<font color=Green> Assignment : Model Experimentation</font>
</h1>

<hr>

<h2>
<font color=Green>Import the Requisite Libraries:</font>
</h2>

In [ ]:
# ignore the warnings in the output
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# import libraries and packages
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import mlflow
from pycaret.classification import *

<hr>
<h2>
<font color=Green>Data Reading:</font>
</h2>

In [ ]:
%%time

##Copy the cleaned data obtained from the data_cleaning notebook into the Data folder before proceeding
dataset = pd.read_csv('data/cleaned_data.csv')

In [ ]:
dataset.head()

In [ ]:
# drop the data column as it is not needed for training
dataset = dataset.drop(['created_date'], axis=1)

In [ ]:
dataset.info()

In [ ]:
# Convert the `city_tier` as categorical type variable, since it was categorical originally
dataset['city_tier'] = dataset.city_tier.astype('category')

<hr>
<h2>
<font color=Green>Set up Database: </font>
</h2>

In [ ]:
# create a connection and setup a SQLite database with the name "lead_scoring_model_experimentation.db" in 
# 'Assignment/02_training_pipeline/notebook/' location
import sqlite3
from sqlite3 import Error

def create_connection(db_file):
    """ create a database connection to a SQLite database """
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        print(sqlite3.version)
    except Error as e:
        print(e)
    finally:
        if conn:
            conn.close()

if __name__ == '__main__':
    create_connection("lead_scoring_model_experimentation.db")

In [ ]:
# Once, your server is successfully running, create a mlflow tracking uri at "http://0.0.0.0:6006"
mlflow.set_tracking_uri("http://0.0.0.0:6006")

In [ ]:
mlflow.set_experiment('Lead_scoring_mlflow_experimentation')

<hr>
<h2>
<font color=Green>Model Experimentation with pycaret: </font>
</h2>

In [ ]:
# setup pycaret 
exp_lead_scoring = setup(data=dataset, target = 'app_complete_flag',
                         fold_shuffle=True, 
                         session_id = 42,
                         normalize = True, 
                         transformation = True, 
                         remove_multicollinearity = True, multicollinearity_threshold = 0.95,
                         n_jobs=-1,use_gpu=True,
                         log_experiment=True,experiment_name='Lead_scoring_model_experimentation',
                         log_plots=True, log_data=True,
                         silent=True, verbose=True,
                         log_profile=False)


In [ ]:
# create a experimentation with pycaret and exclude ['gbc','knn','qda', 'dummy', 'svm', 'ada']
best_model = compare_models(fold = 10, exclude=['gbc','knn','qda', 'dummy', 'svm', 'ada'])

In [ ]:
# create a model which gives the highest accuracy
lightgbm = create_model('lightgbm', fold=10)

In [ ]:
# create feature importance plot
plot_model(lightgbm, plot='feature_all')

In [ ]:
# list of the features that needs to be there in the final encoded dataframe
ONE_HOT_ENCODED_FEATURES = ['total_leads_droppped', 'city_tier_3.0',
                            'city_tier_1.0','first_platform_c_Level0',
                            'city_tier_2.0', 'first_platform_c_others',
                            'first_utm_source_c_others', 'first_utm_medium_c_others',
                            'first_platform_c_Level3', 'referred_lead',
                            'first_platform_c_Level1', 'first_utm_medium_c_Level15',
                            'first_utm_medium_c_Level0', 'first_utm_medium_c_Level11',
                            'first_utm_source_c_Level0', 'first_utm_source_c_Level7',
                            'first_platform_c_Level2', 'first_utm_source_c_Level16',
                            'first_utm_medium_c_Level30', 'first_platform_c_Level7',
                            'first_platform_c_Level8', 'first_utm_medium_c_Level6',
                            'first_utm_medium_c_Level4', 'first_utm_medium_c_Level8',
                            'first_utm_source_c_Level2', 'first_utm_source_c_Level6',
                            'first_utm_medium_c_Level2', 'first_utm_source_c_Level4',
                            'first_utm_medium_c_Level9', 'first_utm_medium_c_Level3',
                            'first_utm_medium_c_Level26', 'app_complete_flag']

In [ ]:
FEATURES_TO_ENCODE = ['city_tier', 'first_platform_c', 'first_utm_medium_c', 'first_utm_source_c']

In [ ]:
df_encoded = pd.DataFrame(columns=ONE_HOT_ENCODED_FEATURES)
df_placeholder = pd.DataFrame()

In [ ]:
# encode the features using get_dummies()
for f in FEATURES_TO_ENCODE:
    if(f in dataset.columns):
        encoded = pd.get_dummies(dataset[f])
        encoded = encoded.add_prefix(f + '_')
        df_placeholder = pd.concat([df_placeholder, encoded], axis=1)
    else:
        print('Feature not found')

In [ ]:
# add the encoded features into a single dataframe
for feature in df_encoded.columns:
    if feature in dataset.columns:
        df_encoded[feature] = dataset[feature]
    if feature in df_placeholder.columns:
        df_encoded[feature] = df_placeholder[feature]
df_encoded.fillna(0, inplace=True)

In [ ]:
df_encoded.head()

In [ ]:
df_encoded.info()

<hr>
<h2>
<font color=Green>Model Experimentation after dropping features: </font>
</h2>

From the above feature tests we can claerly see that some of the features are not significant. We will now drop all the insignificant features and select only the significant ones.
The list of the significant features is 
['total_leads_droppped', 'city_tier', 'referred_lead', 'first_platform_c', 'first_utm_medium_c', 'first_utm_source_c'].
So now you will train your model with onlly these features.

We will drop the following variables
['carrer_interaction'.'assistance_interaction', 'syllabus_interaction', 'social_interaction']

In [ ]:
# Setup Pycaret Enviornment with selected variables 
from pycaret.classification import *

exp_lead_scoring = setup(data=df_encoded , target = 'app_complete_flag',  
                   remove_multicollinearity = True, multicollinearity_threshold = 0.95,
                   fold_shuffle=True, 
                   session_id = 42,
                   n_jobs=-1,use_gpu=True,
                   log_experiment=True,experiment_name='Lead_scoring_mlflow_experimentation',
                   log_plots=True, log_data=True,
                   silent=True, verbose=True,
                   log_profile=False)

In [ ]:
# create a experimentation with pycaret and exclude ['gbc','knn','qda', 'dummy', 'svm', 'ada']. 
best_model = compare_models(fold = 10,exclude=['gbc','knn','qda', 'dummy', 'svm', 'ada'])

In [ ]:
# You should get lightgbm as the best performing model. So now we will train a lightGBM model manually using pycaret
lightgbm_fs = create_model('lightgbm', fold=10)

In [ ]:
#!pip install optuna

In [ ]:
# Tune the hyper parameters of the lightgbm model using optuna on 10 folds and optimise AUC as that was our system metric, 
# hence we will optimise AUC
import optuna

tuned_lgbm_optuna,tuner_1 = tune_model(lightgbm_fs, 
                                   search_library = 'optuna',
                                   fold = 5,
                                   optimize = 'auc',
                                   choose_better = True, 
                                   return_tuner=True)

In [ ]:
# Print the final models configuration so that we can use it in the model retraining pipeline 
print(tuned_lgbm_optuna)

In [ ]:
# Store the selected data columns DataFrame in the SQL Database
cnx = sqlite3.connect("lead_scoring_model_experimentation.db")

df_encoded.to_sql(name='model_experimentation', con=cnx,if_exists='replace', index=False)
cnx.close()